# 📘 The AI Engineer's LLM Workbook

**14 Chapters · 14 Google Colab Notebooks · Beginner to Production**

---

*© 2026 JAWNVION LLC — www.jawnvion.com — peter@jawnvion.com*

*Licensed for individual use. Do not redistribute.*

---

## What's Inside

| # | Chapter |
|---|---------|
| 01 | AI Fundamentals & Problem Framing |
| 02 | Data Science Toolkit (NumPy, Pandas, Matplotlib) |
| 03 | Neural Networks from Scratch |
| 04 | Transformers Architecture Deep Dive |
| 05 | HuggingFace & Pre-Trained Models |
| 06 | QLoRA Fine-Tuning |
| 07 | DPO Alignment Training |
| 08 | Retrieval-Augmented Generation (RAG) |
| 09 | Model Evaluation & Benchmarking |
| 10 | FastAPI Deployment |
| 11 | Monitoring & Observability |
| 12 | Security for AI Systems |
| 13 | Cost Optimization & Quantization |
| 14 | Capstone: End-to-End LLM Project |

---

> **How to use:** Click **Runtime → Run All** in Google Colab, or run cells one at a time.
> Each chapter builds on the last — complete them in order for best results.

---


# Chapter 5: The Hugging Face Ecosystem
**JAWNVION LLC — AI Training Workbook**

Hugging Face is the GitHub of AI models — it hosts 500,000+ models including
TinyLlama, provides the libraries that power 90% of LLM fine-tuning projects,
and gives you a one-line API to load and run any of them.

**What you'll learn:**
- The `transformers` library — AutoTokenizer, AutoModel, pipelines
- The `datasets` library — loading and processing training data
- The `peft` library — LoRA adapters (the foundation of Ch6)
- Load TinyLlama, inspect its architecture and config
- Run your first inference with TinyLlama
- Understand tokens, tokenisation, and why vocabulary size matters
- Save model outputs — bridge to Chapter 6 fine-tuning

**GPU recommended** (T4 is fine). The model loads in < 2 minutes.

In [ ]:
# — Cell 1: Install Packages ——————————————————————————
import subprocess, sys

print("Installing Hugging Face ecosystem...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "tokenizers>=0.22,<0.24"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "transformers", "peft", "accelerate", "datasets", "bitsandbytes",
                "torchao"],
               check=True)

import transformers, peft, datasets, torch

print(f"✓  transformers : {transformers.__version__}")
print(f"✓  peft         : {peft.__version__}")
print(f"✓  datasets     : {datasets.__version__}")
print(f"✓  torch        : {torch.__version__}")
print(f"✓  CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU          : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   VRAM         : {vram:.1f} GB")

In [ ]:
# — Cell 2: Tokenisation — Text to Numbers ————————————
from transformers import AutoTokenizer

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"Loading tokenizer from: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print(f"✓  Tokenizer loaded")
print(f"   Vocabulary size  : {tokenizer.vocab_size:,} tokens")
print(f"   Model max length : {tokenizer.model_max_length}")
print(f"   Padding side     : {tokenizer.padding_side}")
print()

# Tokenise some examples
examples = [
    "The quick brown fox jumps over the lazy dog.",
    "QLoRA fine-tunes large language models with 4-bit quantization.",
    "DocuMind AI processes technical documentation automatically.",
]

print("TOKENISATION EXAMPLES")
print("=" * 60)
for text in examples:
    tokens = tokenizer.encode(text)
    decoded = [tokenizer.decode([t]) for t in tokens]
    print(f"\n  Text    : {text}")
    print(f"  Tokens  : {tokens}")
    print(f"  Decoded : {decoded}")
    print(f"  Length  : {len(tokens)} tokens")

print()
print("Key insight: 'QLoRA' is likely split into ['Q', 'Lo', 'RA'].")
print("The tokenizer was trained on general text — not AI terminology.")
print("This is why domain-specific fine-tuning improves output quality.")

In [ ]:
# — Cell 3: Special Tokens & the Chat Template ————————
from transformers import AutoTokenizer

# Chat models use special tokens to structure conversations.
# The model was trained to expect this exact format.
# Getting it wrong → poor quality output, even with great weights.

print("TINYLLAMA CHAT TEMPLATE")
print("=" * 55)
print()
print("Special tokens:")
print(f"  BOS (beginning of seq) : {tokenizer.bos_token!r} (id={tokenizer.bos_token_id})")
print(f"  EOS (end of seq)       : {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print(f"  PAD (padding)          : {tokenizer.pad_token!r}")
print()

# Format a prompt using the chat template
messages = [
    {"role": "system",    "content": "You are a helpful AI assistant for JAWNVION LLC."},
    {"role": "user",      "content": "What is the difference between fine-tuning and RAG?"},
]

formatted = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print("Formatted chat prompt (what gets sent to the model):")
print("-" * 55)
print(formatted)
print("-" * 55)
token_ids = tokenizer.encode(formatted)
print(f"\nToken count: {len(token_ids)} tokens")
print()
print("Rule: ALWAYS use apply_chat_template for chat models.")
print("In Ch6 (fine-tuning), SFTTrainer handles this automatically.")
print("In Ch10 (serving), the /generate endpoint does it for you.")

In [ ]:
# — Cell 4: Load TinyLlama — Inspect the Architecture ——
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
device   = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading TinyLlama on {device}...")
print("(First run downloads ~2.2 GB — subsequent runs use the Colab cache)")
print()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
)
model.eval()

# Parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓  Model loaded!")
print(f"   Total parameters    : {total_params:,}")
print(f"   Trainable params    : {trainable:,}")
print(f"   dtype               : {next(model.parameters()).dtype}")
if device == "cuda":
    vram_used = torch.cuda.memory_allocated() / 1e9
    print(f"   VRAM used           : {vram_used:.2f} GB")
print()

# Architecture config
cfg = model.config
print("CONFIG (the numbers from Chapter 4 — now real):")
print(f"  hidden_size     : {cfg.hidden_size}")
print(f"  num_layers      : {cfg.num_hidden_layers}")
print(f"  num_attn_heads  : {cfg.num_attention_heads}")
print(f"  num_kv_heads    : {cfg.num_key_value_heads}")
print(f"  intermediate_sz : {cfg.intermediate_size}")
print(f"  max_position_emb: {cfg.max_position_embeddings}")
print(f"  vocab_size      : {cfg.vocab_size:,}")
print(f"  hidden_act      : {cfg.hidden_act}")
print(f"  rope_theta      : {getattr(cfg, 'rope_theta', getattr(cfg, 'rope_scaling', {}) or 10000)}")

In [ ]:
# — Cell 5: Run Your First Inference ——————————————————
import torch, time

def generate(prompt, max_new_tokens=200, temperature=0.7):
    """Generate text from TinyLlama using the chat template."""
    messages = [
        {"role": "system",  "content": "You are a concise, helpful AI assistant."},
        {"role": "user",    "content": prompt},
    ]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    n_input = inputs["input_ids"].shape[1]

    t0 = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.time() - t0

    new_tokens = outputs[0][n_input:]
    response   = tokenizer.decode(new_tokens, skip_special_tokens=True)
    tps        = len(new_tokens) / elapsed
    return response, len(new_tokens), elapsed, tps

# Run 3 test prompts
prompts = [
    "In one sentence, what is gradient descent?",
    "What is the difference between fine-tuning and prompting an LLM?",
    "List three advantages of using INT4 quantization for model deployment.",
]

print("TINYLLAMA INFERENCE RESULTS")
print("=" * 60)
for i, prompt in enumerate(prompts, 1):
    print(f"\n[{i}] Q: {prompt}")
    response, n_tok, elapsed, tps = generate(prompt, max_new_tokens=150)
    print(f"    A: {response.strip()}")
    print(f"    [{n_tok} tokens in {elapsed:.1f}s — {tps:.1f} tok/s]")

print()
print("✓  TinyLlama is working!")
print("   In Chapter 6 you'll FINE-TUNE this same model on your own data.")
print("   In Chapter 10 you'll wrap this generate() function in a REST API.")

In [ ]:
# — Cell 6: The Pipelines API — One-Line Inference ————
from transformers import pipeline

# The pipeline API abstracts tokenise → generate → decode into one call.
# Use it for quick experimentation; use AutoModel directly for production.

print("HUGGING FACE PIPELINES API")
print("=" * 50)
print()

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
)

result = generator(
    "Explain QLoRA in exactly two sentences.",
    max_new_tokens=100,
    temperature=0.6,
    do_sample=True,
    return_full_text=False,
)

print("Pipeline result:")
print(result[0]["generated_text"].strip())
print()
print("Available pipeline tasks:")
tasks = {
    "text-generation":     "Generate text (what we use for LLMs)",
    "text-classification": "Sentiment / intent classification",
    "token-classification":"Named-entity recognition (NER)",
    "question-answering":  "Extractive QA — finds answer span in context",
    "summarization":       "Abstractive text summarisation",
    "translation":         "Language translation",
    "fill-mask":           "Masked language modelling (BERT-style)",
    "feature-extraction":  "Get embeddings (used in RAG, Ch8)",
}
for task, desc in tasks.items():
    marker = "← we use this" if task in ["text-generation","feature-extraction"] else ""
    print(f"  {task:<26}  {desc}  {marker}")

In [ ]:
# — Cell 7: PEFT — Preview of Chapter 6 Fine-Tuning ——
from peft import LoraConfig, get_peft_model, TaskType

print("PEFT (Parameter-Efficient Fine-Tuning) — LORA PREVIEW")
print("=" * 55)
print()
print("Fine-tuning ALL 1.1B parameters is expensive (requires 40+ GB VRAM).")
print("LoRA adds SMALL trainable matrices alongside the frozen original weights.")
print()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                      # rank — controls adapter size
    lora_alpha=32,             # scaling factor
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none",
)

peft_model = get_peft_model(model, lora_config)

total  = sum(p.numel() for p in peft_model.parameters())
trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
pct = trainable / total * 100

print(f"LoRA configuration:")
print(f"  rank (r)          : {lora_config.r}")
print(f"  alpha             : {lora_config.lora_alpha}")
print(f"  target modules    : {lora_config.target_modules}")
print()
print(f"Parameter counts:")
print(f"  Total parameters  : {total:>15,}")
print(f"  Trainable (LoRA)  : {trainable:>15,}  ({pct:.2f}%)")
print(f"  Frozen (base)     : {total-trainable:>15,}")
print()
print(f"✓  We train only {pct:.2f}% of the model — that's the QLoRA magic.")
print(f"   Chapter 6 will add 4-bit quantisation on top of this,")
print(f"   reducing VRAM from ~2.2 GB (fp16) to ~0.7 GB (INT4 + LoRA).")

# Clean up PEFT model — don't want it competing with Ch6
del peft_model
import gc
gc.collect()
if __import__('torch').cuda.is_available():
    __import__('torch').cuda.empty_cache()

In [ ]:
# — Cell 8: Save Model State — Bridge to Chapter 6 ————
import os, shutil

# Verify the base model is loaded and accessible.
# Chapter 6's Cell 4 will re-load it from HuggingFace cache automatically —
# no need to save it manually. But we'll confirm the cache path.

cache_dir = os.path.expanduser("~/.cache/huggingface/hub")
model_dirs = []
if os.path.exists(cache_dir):
    for d in os.listdir(cache_dir):
        if "TinyLlama" in d:
            model_dirs.append(d)

print("CHAPTER 5 → CHAPTER 6 BRIDGE")
print("=" * 50)
print()
print("Hugging Face cache (model weights are stored here):")
for d in model_dirs:
    full = os.path.join(cache_dir, d)
    size = sum(
        os.path.getsize(os.path.join(root, f))
        for root, _, files in os.walk(full)
        for f in files
    ) / 1e9
    print(f"  {d}  ({size:.2f} GB)")

if not model_dirs:
    print("  (cache not found — model was loaded differently in this session)")
    print("  Chapter 6 will auto-download from HuggingFace Hub.")

print()
print("What Chapter 6 (QLoRA Fine-Tuning) builds on top of this:")
print()
steps = [
    "1. Load TinyLlama in 4-bit (INT4/NF4) — uses 0.7 GB VRAM instead of 2.2 GB",
    "2. Attach LoRA adapters to Q,K,V,O projections (r=16, alpha=32)",
    "3. Prepare a formatted Q&A dataset with the chat template from Cell 3",
    "4. Train with SFTTrainer for 3 epochs — only LoRA weights update",
    "5. Save adapter → merge → export merged model for serving (Ch10)",
]
for step in steps:
    print(f"  {step}")

print()
print("✓  Chapter 5 complete. You're ready to fine-tune.")

In [ ]:
# — Cell 9: Hugging Face Ecosystem Map ——————————————
ecosystem = {
    "transformers": {
        "what":     "Load and run 500k+ pretrained models",
        "key APIs": "AutoTokenizer, AutoModel, pipeline, Trainer",
        "used in":  "Ch5 (load), Ch6 (fine-tune), Ch8 (embeddings), Ch10 (serve)",
    },
    "peft": {
        "what":     "Parameter-Efficient Fine-Tuning (LoRA, QLoRA, prefix tuning)",
        "key APIs": "LoraConfig, get_peft_model, PeftModel.from_pretrained",
        "used in":  "Ch6 (QLoRA), Ch7 (DPO adapter), Ch10 (merge_and_unload)",
    },
    "datasets": {
        "what":     "Load, stream, and process any ML dataset",
        "key APIs": "load_dataset, .map, .filter, .train_test_split",
        "used in":  "Ch2 (EDA), Ch6 (training data), Ch8 (RAG corpus)",
    },
    "accelerate": {
        "what":     "Multi-GPU / mixed-precision training abstraction",
        "key APIs": "Accelerator, device_map='auto', BitsAndBytesConfig",
        "used in":  "Ch6-14 (enables 4-bit loading with device_map=auto)",
    },
    "bitsandbytes": {
        "what":     "INT8/INT4 quantization for loading large models on small GPUs",
        "key APIs": "BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4')",
        "used in":  "Ch6 (QLoRA), Ch13 (benchmarking), Ch14 (capstone)",
    },
    "trl": {
        "what":     "Transformer Reinforcement Learning — SFT, DPO, PPO trainers",
        "key APIs": "SFTTrainer, DPOTrainer, DPOConfig",
        "used in":  "Ch6 (SFTTrainer), Ch7 (DPOTrainer)",
    },
    "sentence-transformers": {
        "what":     "Sentence and document embeddings for similarity / retrieval",
        "key APIs": "SentenceTransformer, .encode, cosine_similarity",
        "used in":  "Ch8 (RAG indexing), Ch14 (Capstone FAISS index)",
    },
}

print("HUGGING FACE ECOSYSTEM — COURSE REFERENCE")
print("=" * 65)
for lib, info in ecosystem.items():
    print(f"\n  [{lib}]")
    for k, v in info.items():
        print(f"    {k:<10} : {v}")

## ✓ Chapter 5 Complete

TinyLlama is loaded, tokenised, and generating text on your T4 GPU.
You understand the full Hugging Face toolkit and how each library connects
to the chapters ahead.

**Next:** Chapter 6 — QLoRA Fine-Tuning
You'll fine-tune TinyLlama on a domain-specific Q&A dataset using QLoRA —
4-bit quantisation + LoRA adapters — training only 0.6% of the model's
parameters while preserving 95%+ of response quality.

---
*JAWNVION LLC AI Training Workbook · peter@jawnvion.com*